In [ ]:
"""
git clone https://github.com/quinnlue/asr.git
cd asr
sudo apt update
sudo apt install just
just create-environment
source ./.venv/bin/activate
just requirements
uv add --dev ipykernel
uv run ipython kernel install --user --env VIRTUAL_ENV $(pwd)/.venv --name=asr
uv run --with jupyter jupyter lab
Available types: ['speed', 'time_stretch', 'gain', 'silence', 'impulse', 'shift', 'noise', 'noise_norm', 'white_noise', 'rir_noise_aug', 'transcode_aug', 'random_segment']
"""

In [ ]:
# Standard library
import json
import os
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
from matplotlib import ticker

# Core ML & audio stack
import librosa
import lightning.pytorch as pl
import numpy as np
import torch
from tqdm.auto import tqdm

# ASR models & normalization
from nemo.collections.asr.models import ASRModel
from transformers.models.whisper.english_normalizer import EnglishTextNormalizer

# Training & experiment utilities
from loguru import logger
from nemo.utils import logging
from nemo.utils.exp_manager import exp_manager
from nemo.utils.trainer_utils import resolve_trainer_cfg
from omegaconf import OmegaConf, open_dict

# Project utilities
from asr_benchmark.config import PROJECT_ROOT
from asr_benchmark.manifest import prepare_manifests
from asr_benchmark.nemo_adapter import (
    add_global_adapter_cfg,
    patch_transcribe_lhotse,
    update_model_cfg,
    update_model_config_to_support_adapter,
)
from asr_benchmark.score import english_spelling_normalizer, score_wer

In [ ]:
torch.set_float32_matmul_precision("high")
# Set SAMPLE to use a smaller subset of the data for faster iteration during development. Set it to None to use the full dataset.
SAMPLE = None

In [ ]:
manifests = prepare_manifests(sample_n=SAMPLE)
train_manifest_path = manifests["train"]
val_manifest_path = manifests["val"]
noise_manifest_path = manifests["noise"]
impulse_manifest_path = manifests["impulse"]

In [ ]:
# ── Hardware-dependent settings ──────────────────────────────────────────────
# Adjust these to match your GPU memory and CPU cores.
DEVICES = 1
PRECISION = "bf16-mixed"
BATCH_SIZE = 32
NUM_WORKERS = 8

# ── Model settings ───────────────────────────────────────────────────────────
PRETRAINED_MODEL = "nvidia/parakeet-tdt-0.6b-v3"
ADAPTER_NAME = "asr_children_orthographic"
ADAPTER_MODULE = "encoder"
ADAPTER_TYPE = "lora"
ENCODER_DIM = 1024
LORA_RANK = 16

# ── Training settings ────────────────────────────────────────────────────────
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.0
MAX_EPOCHS = 10
GRAD_ACCUMULATION = 2
VAL_CHECK_INTERVAL = 1.0  # validate every N epochs (float) or steps (int)
CHANNEL_SELECTOR = "average"

# ── Augmentation settings ────────────────────────────────────────────────────
TIME_STRETCH_PROB = 0.5
NOISE_PROB = 0.5
IMPULSE_PROB = 0.5
AUG_RNG = 42
SAMPLE_RATE = 16000
MIN_SNR_DB = 10.0
MAX_SNR_DB = 50.0
MIN_SPEED_RATE = 0.9
MAX_SPEED_RATE = 1.1
NUM_SPEED_RATES = 5

# ── Spec augmentation settings ───────────────────────────────────────────────
FREQ_MASKS = 2
TIME_MASKS = 2
FREQ_WIDTH = 27
TIME_WIDTH = 0.05

# ── Experiment tracking ──────────────────────────────────────────────────────
WANDB_PROJECT = "asr-ctc"
WANDB_RUN_NAME = "orthographic-adapter"
WANDB_TAGS = ["orthographic", "adapter", PRETRAINED_MODEL]
EXP_DIR = str(PROJECT_ROOT / "models" / "orthographic_benchmark_nemo")

# ── Load NeMo adapter defaults ───────────────────────────────────────────────
yaml_path = PROJECT_ROOT / "asr_benchmark" / "assets" / "asr_adaptation.yaml"
cfg = OmegaConf.load(yaml_path)

# ── Training overrides ───────────────────────────────────────────────────────
overrides = OmegaConf.create(
    {
        "model": {
            "pretrained_model": PRETRAINED_MODEL,
            "adapter": {
                "adapter_name": ADAPTER_NAME,
                "adapter_module_name": ADAPTER_MODULE,
                "adapter_type": ADAPTER_TYPE,
                ADAPTER_TYPE: {"in_features": ENCODER_DIM, "rank": LORA_RANK},
            },
            "train_ds": {
                "manifest_filepath": str(train_manifest_path),
                "batch_size": BATCH_SIZE,
                "num_workers": NUM_WORKERS,
                "use_lhotse": False,
                "channel_selector": CHANNEL_SELECTOR,
                "augmentor": {
                    "time_stretch": {
                        "min_speed_rate": MIN_SPEED_RATE,
                        "max_speed_rate": MAX_SPEED_RATE,
                        "prob": TIME_STRETCH_PROB,
                        "num_rates": NUM_SPEED_RATES,
                        "rng": AUG_RNG,
                    },
                    "noise": {
                        "manifest_path": str(noise_manifest_path),
                        "orig_sr": SAMPLE_RATE,
                        "min_snr_db": MIN_SNR_DB,
                        "max_snr_db": MAX_SNR_DB,
                        "rng": AUG_RNG,
                        "prob": NOISE_PROB,
                    },
                    "impulse": {
                        "manifest_path": str(impulse_manifest_path),
                        "rng": AUG_RNG,
                        "prob": IMPULSE_PROB,
                    },
                },
            },
            "validation_ds": {
                "manifest_filepath": str(val_manifest_path),
                "batch_size": BATCH_SIZE,
                "num_workers": NUM_WORKERS,
                "use_lhotse": False,
                "channel_selector": CHANNEL_SELECTOR,
            },
            "spec_augment": {
                "_target_": "nemo.collections.asr.modules.SpectrogramAugmentation",
                "freq_masks": FREQ_MASKS,
                "time_masks": TIME_MASKS,
                "freq_width": FREQ_WIDTH,
                "time_width": TIME_WIDTH,
            },
            "optim": {
                "lr": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
            },
        },
        "trainer": {
            "devices": DEVICES,
            "precision": PRECISION,
            "strategy": "auto",
            "max_epochs": 1 if SAMPLE else MAX_EPOCHS,
            "max_steps": -1,
            "accumulate_grad_batches": GRAD_ACCUMULATION,
            "val_check_interval": VAL_CHECK_INTERVAL,
            "enable_progress_bar": False,
        },
        "exp_manager": {
            "exp_dir": EXP_DIR,
            "create_wandb_logger": True,
            "wandb_logger_kwargs": {
                "project": WANDB_PROJECT,
                "name": WANDB_RUN_NAME,
                "tags": WANDB_TAGS,
            },
        },
    }
)

cfg = OmegaConf.merge(cfg, overrides)

## 3. Define Trainer

The Trainer orchestrates the training loop across devices, delegating tensor operations to PyTorch's backend. We initiate the trainer with the `OmegaConf` config object we made above, and then set up an experiment manager to handle logging, checkpoint saving, and saving artifacts to disk.

Note we use the cell magic [`%%capture`](https://ipython.readthedocs.io/en/9.2.0/interactive/magics.html#cellmagic-capture) below to hide long cell outputs for readability.

In [ ]:
trainer = pl.Trainer(**resolve_trainer_cfg(cfg.trainer))
exp_log_dir = exp_manager(trainer, cfg.get("exp_manager", None))

In [ ]:
model_cfg = ASRModel.from_pretrained(cfg.model.pretrained_model, return_config=True)
update_model_config_to_support_adapter(model_cfg, cfg)
model = ASRModel.from_pretrained(
    cfg.model.pretrained_model,
    override_config_path=model_cfg,
    trainer=trainer,
)

In [ ]:
with open_dict(model.cfg):
    model.cfg.decoding.greedy.use_cuda_graph_decoder = False
model.change_decoding_strategy(model.cfg.decoding)

In [ ]:
cfg.model.train_ds = update_model_cfg(model.cfg.train_ds, cfg.model.train_ds)
model.setup_training_data(cfg.model.train_ds)

cfg.model.validation_ds = update_model_cfg(model.cfg.validation_ds, cfg.model.validation_ds)
model.setup_multiple_validation_data(cfg.model.validation_ds)

In [ ]:
model.setup_optimization(cfg.model.optim)

In [ ]:
# Configure spec augmentation from config if available, otherwise disable it.
if "spec_augment" in cfg.model:
    model.spec_augmentation = model.from_config_dict(cfg.model.spec_augment)
else:
    model.spec_augmentation = None
    del model.cfg.spec_augment

In [ ]:
with open_dict(cfg.model.adapter):
    adapter_name = cfg.model.adapter.pop("adapter_name")
    adapter_type = cfg.model.adapter.pop("adapter_type")
    adapter_module_name = cfg.model.adapter.pop("adapter_module_name", None)
    adapter_state_dict_name = cfg.model.adapter.pop("adapter_state_dict_name", None)

    adapter_type_cfg = cfg.model.adapter[adapter_type]

    if adapter_module_name is not None and ":" not in adapter_name:
        adapter_name = f"{adapter_module_name}:{adapter_name}"

    adapter_global_cfg = cfg.model.adapter.pop(model.adapter_global_cfg_key, None)
    if adapter_global_cfg is not None:
        add_global_adapter_cfg(model, adapter_global_cfg)

model.add_adapter(adapter_name, cfg=adapter_type_cfg)
assert model.is_adapter_available()

model.set_enabled_adapters(enabled=False)
model.set_enabled_adapters(adapter_name, enabled=True)

model.freeze()
model = model.train()
model.unfreeze_enabled_adapters()

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")

In [ ]:
%%capture
trainer.fit(model)
wandb.finish()

In [ ]:
if adapter_state_dict_name is not None:
    state_path = exp_log_dir if exp_log_dir is not None else os.getcwd()
    ckpt_path = os.path.join(state_path, "checkpoints")
    if os.path.exists(ckpt_path):
        state_path = ckpt_path
    state_path = os.path.join(state_path, adapter_state_dict_name)

    model.save_adapters(str(state_path))

In [ ]:
nemo_ckpts = sorted((exp_log_dir / "checkpoints").glob("*.nemo"))
if not nemo_ckpts:
    raise FileNotFoundError(f"No .nemo checkpoints found in {exp_log_dir}/checkpoints/")

best_ckpt = nemo_ckpts[-1]
print(f"Loading checkpoint: {best_ckpt}")
eval_model = ASRModel.restore_from(best_ckpt, map_location="cuda")

with open_dict(eval_model.cfg):
    eval_model.cfg.decoding.greedy.use_cuda_graph_decoder = False
eval_model.change_decoding_strategy(eval_model.cfg.decoding)

patch_transcribe_lhotse(eval_model)

In [ ]:
val_manifest_path = cfg.model.validation_ds.manifest_filepath
with open(val_manifest_path) as f:
    val_entries = [json.loads(line) for line in f]

audio_files = [e["audio_filepath"] for e in val_entries]
references = [e["text"] for e in val_entries]

print(f"Running inference on {len(audio_files)} validation utterances...")
raw = eval_model.transcribe(
    audio_files, batch_size=BATCH_SIZE, channel_selector="average", verbose=False
)
if isinstance(raw, tuple):
    raw = raw[0]

predictions = [h.text if hasattr(h, "text") else h for h in raw]

In [ ]:
normalizer = EnglishTextNormalizer(english_spelling_normalizer)
filtered = [(r, p) for r, p in zip(references, predictions) if normalizer(r) != ""]

references, predictions = zip(*filtered)

wer = score_wer(references, predictions)

print(f"Validation WER: {wer:.4f}")

print("\nSample predictions:")
for ref, pred in zip(references[:5], predictions[:5]):
    print(f"  REF:  {ref}")
    print(f"  PRED: {pred}")
    print()